# Reed-Solomon Decoder

In [1]:
%load_ext autoreload
%autoreload 1
%aimport classes.galois_field
%aimport classes.galois_poly

from classes.galois_field import *
from classes.galois_poly import *

In [61]:
gf_extended = GaloisField(4, 0b10011)
root        = [GFE1, 1, 2, 3, 4, 5]

def do_unpack(value, degree, gf):
    mask    = gf._order
    ak      = [(value >> i*gf._field_m) & mask for i in range(degree+1)]
    ak      = [gf._get_power(a) for a in ak]
    return GaloisPoly(gf, ak[::-1])

def get_discrepancy(mu, syn, Lx):
    d_mu    = syn[mu]
    for i in range(1, Lx._degree+1):
        term    = gf_extended.mul(Lx @ i, syn[mu - i])[ID_POWER]
        d_mu    = gf_extended.add(d_mu, term)[ID_POWER]
    return d_mu

def get_locator_poly(syn):
    Lx      = []
    mu_lmu  = []
    dmu     = []

    # initialization
    poly = do_unpack(0x1, 0, gf_extended)
    Lx.append(poly)

    for mu in range(2*t):
        mu_lmu.append(mu - Lx[-1]._degree)

        disc = get_discrepancy(mu, syn, Lx[-1])
        # if dicrepancy zero, locator adjust not needed
        if disc == GFE0:
            poly = Lx[-1]
        else:
            if mu == 0:
                # first adjustment: L1 = 1 + d0*x always (because dm1 = 1)
                xm      = do_unpack(0x10, 1, gf_extended)
                poly    = Lx[-1] + (xm * disc)
            else:
                # to adjust we use all previous rho, except newly calculated
                rho     = mu_lmu.index(max(mu_lmu[:-1]))
                xm      = [GFE1] + (mu - rho)*[GFE0]
                xm      = GaloisPoly(gf_extended, xm)
                xm      = xm * gf_extended.div(disc, dmu[rho])[ID_POWER]
                poly    = Lx[-1] + (xm * Lx[rho])

        dmu.append(disc)
        Lx.append(poly)

        print(Lx[-1], "\n")

Field Closed Succesfully!, 15 Non-Zero Elements


## Get Syndromes

In [59]:
r   = 0x52D0A_F17CB_1EE64
# r   = 0x56D09_F171B_1E06F
r   = do_unpack(r, 14, gf_extended)

syn = [r(a) for a in root]

syn


[13, 2, 7, 10, 9, 5]

## Berlekamp-Massey Algorithm

In [63]:
t = 3

get_locator_poly(syn)


a^13*x + 1 

a^4*x + 1 

a^12*x^2 + a^4*x + 1 

a^12*x^2 + a^4*x + 1 

a^12*x^2 + a^4*x + 1 

a^12*x^2 + a^4*x + 1 

